In [10]:
# Authentication & Google Drive-free version of the below cells, uncomment if there are problems
# COLAB ONLY CELLS
try:
   import google.colab
   IN_COLAB = True
   !pip3 install transformers  # https://huggingface.co/docs/transformers/installation
#    !nvidia-smi                 # Check which GPU has been chosen for us
   !rm -rf logs
   # Download the dataset from personal drive
   !mkdir data
   !wget --no-check-certificate 'https://docs.google.com/uc?export=download&id=19jcMX4KFwVAp4yvgvw1GXSnSgpoQytqg' -O data/training_set.json
except:
   IN_COLAB = False

In [11]:
%load_ext tensorboard

import os
import requests
import zipfile
from tqdm import tqdm
import time
import random
import datetime
from IPython.display import display
from functools import partial

from typing import List, Dict, Callable, Sequence, Tuple

from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

%matplotlib inline

# Fix random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

############################## CONFIG ###########################

SMALL_TRAIN_LEN = 20 # NUMBER OF ARTICLES
SMALL_VAL_LEN = 5
BATCH_SIZE = 16 # NUMBER OF PARAGRAPH + QUESTION PAIRS
VAL_BATCH_SIZE = 4

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [12]:
TRAINING_FILE = os.path.join('..','data', 'training_set.json')
with open(TRAINING_FILE, 'r') as f:
    questions = json.load(f)

In [13]:
TRAIN_SPLIT_ELEM = int(len(questions['data']) / 100 * 75)
print("Using {} articles for validation and {} for training".format(
    len(questions['data']) - TRAIN_SPLIT_ELEM, TRAIN_SPLIT_ELEM)
)

data = random.sample(questions['data'], len(questions['data'])) # reshuffle the samples

Using 111 articles for validation and 331 for training


In [14]:
train_dataset = {'data': data[:TRAIN_SPLIT_ELEM]} # recreate the original dataset structure lost by shuffling through the dictionary
val_dataset = {'data': data[TRAIN_SPLIT_ELEM:]}

# we also create a small training set to test the model while building it, just to speed up

small_data = random.sample(train_dataset["data"], SMALL_TRAIN_LEN)
small_train_dataset = {'data': small_data}
small_val_data = random.sample(val_dataset["data"], SMALL_VAL_LEN)
small_val_dataset = {'data': small_val_data}

In [15]:
from transformers import DistilBertTokenizerFast, TFDistilBertModel
# We are using a tokenizer that derives from a "normal" (and not "large") BERT model
# moreover, it ignores casing (uncased)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased') # TODO: cased or uncased?


In [16]:
MAX_LEN_PAIRS = 512

def find_start_end_token_one_hot_encoded(answers: Dict, offsets: List[Tuple[int]]) -> int:
    '''
    This function returns the starting and ending token of the answer, already one hot encoded and ready for binary crossentropy
    Inputs:
        answers: List[Dict] --> for each question, a list of answers. Each answer contains:
            - answer_start: the index of the starting character
            - text: the text of the answer, that we exploit through the number of chars that it containts
        offsets: List[Tuple[int]] --> the tokenizer from HuggingFace transforms the sentence (question+context)
            into a sequence of tokens. Offsets keeps track of the character start and end indexes for each token
    Output:
        result: Dict --> each key contains only one array, the one-hot encoded version of, respectively, the start
            and end token of the answer in the sentence (question+context)
    '''
    result = {
        "out_S": np.zeros(len(offsets)),
        "out_E": np.zeros(len(offsets))
    }   

    for answer in answers:
        starting_char = answer['answer_start']
        answer_len = len(answer['text'])

        for i in range(1, len(offsets)): # we skip the first token, [CLS], that has (0,0) as a tuple
            # We cycle through all the tokens of the question, until we find (0,0), which determines the separator
            if offsets[i] == (0,0): # The [SEP] special char --> this indicates the beginning of the context
                for j in range(1, len(offsets)-i-1): # We skip the first and the last tokens, both special tokens
                    # If the starting char is in the interval, the index (j) of its position inside the context, 
                    # plus the length of the question (i) is the right index
                    if (starting_char >= offsets[i+j][0]) and (starting_char <= offsets[i+j][1]):
                        result["out_S"][i+j] += 1
                    # if the ending char (starting + length -1) is in the interval, same as above
                    if (starting_char + answer_len - 1 >= offsets[i+j][0]) and (starting_char + answer_len - 1 < offsets[i+j][1]):
                        result["out_E"][i+j] += 1
                        break
                # After this cycle, we must check other answers
                break
    
    return result

def create_data_for_dataset(data, max_pairs_length=512):
    '''
    This function takes in input the whole data structure and iteratively composes question+context pairs, plus their label
    Inputs:
        data: Dict --> the data structure containing the data
    Outputs:
        tf.data.Dataset --> the data structure containing (features, labels) that will be fed to the model during fitting
        more specifically:
        features: Dict --> keys:
            - input_ids: array of token ids
            - attention_mask: array indicating if the corresponding token is padding or not
        labels: Dict --> keys:
            - gt_S: array representing the index of the initial token of the answer, one-hot encoded
            - gt_E: array representing the index of the final token of the answer, one-hot encoded

    This function, for each article in "data", extracts all paragraphs (and their text, the "context"), for each paragraph, all questions_and_answers
    At this point, it tokenizes (question+context) while truncating and padding up to MAX_LEN_PAIRS
    Moreover, it also returns the "attention_mask", an array that tells if the token is padding or normal, that will be used by the model

    It also keeps track, through "find_start_end_token_one_hot_encoded", of the index of the initial and final token of the answer, the labels for the model

    In the end, it returns a tf.data.Dataset with the structure (features, labels), to be injected directly in the fit method of the model
    '''
    features = []
    labels = []

    for article in tqdm(data["data"]):
        for paragraph in article["paragraphs"]:
            for question_and_answer in paragraph["qas"]:
                ### QUESTION AND CONTEXT TOKENIZATION ###
                # For question answering with BERT we need to encode both 
                # question and context, and this is the way in which 
                # HuggingFace's BertTokenizer does it.
                # The tokenizer returns a dictionary containing all the information we need
                encoded_inputs = tokenizer(
                    question_and_answer["question"],    # First we pass the question
                    paragraph["context"],               # Then the context
                    max_length =  max_pairs_length,         # We want to pad and truncate to this length
                    truncation = True,
                    padding = 'max_length',             # Pads all sequences to 512.
                                                        # If "True" it would pad to the longest sentence in the batch 
                                                        # (in this case we only use 1 sentence, so no padding at all)
                    # return_token_type_ids = True,     # IF USING BERT, DistilBert does not need it 
                    return_token_type_ids = False,      # Return if the token is from sentence 0 or sentence 1 
                    return_attention_mask = True,       # Return if it's a pad token or not
                    return_offsets_mapping = True       # Really important --> returns each token's first and last char position in the original sentence 
                )
                
                ### MAPPING OF THE START OF THE ANSWER BETWEEN CHARS AND TOKENS ###
                # We want to pass from the starting position in chars to the starting position in tokens
                label = find_start_end_token_one_hot_encoded(
                    # We pass the list of answers (usually there is still one per question,
                    #   but we mustn't assume anything)
                    answers = question_and_answer["answers"],
                    # And also the inputs offset mapping just recieved from the tokenizer
                    offsets = encoded_inputs["offset_mapping"]
                )
                
                encoded_inputs.pop("offset_mapping", None) # Removes the offset mapping, not useful anymore 
                                                           # ("None" is used because otherwise KeyError could be raised if the key wasn't present)
                features.append(encoded_inputs)
                labels.append(label)

                # DO NOT KNOW IF IT IS NEEDED
                '''
                ### ANSWER TOKENIZATION ###
                # use the same tokenizer also to tokenize the answers
                encoded_answer = tokenizer(
                    question_and_answer["answers"][0]["text"],  # here we only need to pass the answer
                    max_length=MAX_LEN_ANSWERS,
                    truncation = True,
                    padding = 'max_length',
                    add_special_tokens = False,                 # the answer will only be used for the loss, not as input to the model, it does not need special tokens [CLS] and [SEP]
                    return_token_type_ids = False,              # only one sentence
                    return_attention_mask = True)               # still interested in padding tokens
                
                # we add to the dictionary of the pair question-context the token ids of the answer and its mask
                encoded_inputs["answer_ids"] = encoded_answer["input_ids"]
                encoded_inputs["answer_mask"] = encoded_answer["attention_mask"]
                '''

    print("Creating dataset")
    return tf.data.Dataset.from_tensor_slices((
        pd.DataFrame.from_dict(features).to_dict(orient="list"),  # dataframe for features 
        pd.DataFrame.from_dict(labels).to_dict(orient="list")                                                    # dataframe for labels 
    ))

In [17]:
# create the datasets
#####################################################################
 ######## TODO: CHANGE LINE BELOW WITH "train_dataset" #############
#####################################################################
train_ds = create_data_for_dataset(small_train_dataset)
val_ds = create_data_for_dataset(small_val_dataset)

100%|██████████| 20/20 [00:07<00:00,  2.81it/s]


Creating dataset


100%|██████████| 5/5 [00:00<00:00,  5.64it/s]


Creating dataset


In [18]:
# OSS: remember to call fit with this structure {"input_ids": train_df["input_ids"], ...}
# Load the model pretrained on the masked input + sentence completion task
transformer_model = TFDistilBertModel.from_pretrained("distilbert-base-uncased", output_hidden_states = True) # add here config for the model, ex. also attention outputs

Some layers from the model checkpoint at distilbert-base-uncased were not used when initializing TFDistilBertModel: ['vocab_projector', 'vocab_transform', 'vocab_layer_norm', 'activation_13']
- This IS expected if you are initializing TFDistilBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFDistilBertModel were initialized from the model checkpoint at distilbert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


In [19]:
def create_model(max_pairs_length=512):
    input_ids = tf.keras.Input(shape=(max_pairs_length, ), name="input_ids", dtype='int32')
    attention_mask = tf.keras.Input(shape=(max_pairs_length, ), name="attention_mask", dtype='int32')
    # token_type_ids = tf.keras.Input(shape=(SHAPE_ATTENTION_MASK, ), dtype='int32') # uncomment if using BERT

    transformer = transformer_model(
        {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            # "token_type_ids": token_type_ids # uncomment if using BERT
        }
    )
    print(transformer)
    # TODO: chose which layers
    hidden_states = transformer.hidden_states
    chosen_states_idx = [3, 4, 5, 6]

    # TODO: chose merging method
    chosen_hidden_states = layers.concatenate(
        tuple([hidden_states[i] for i in chosen_states_idx])
    )

    # output = layers.Bidirectional(layers.LSTM(64, return_sequences = True, activation = "relu"))(chosen_hidden_states)
    # output = layers.Dense(2, activation = "softmax")(output) # 2 because we need both 

    out_S = layers.Dense(1)(chosen_hidden_states) # dot product between token representation and start vector
    out_S = layers.Flatten()(out_S)
    out_S = layers.Softmax(name="out_S")(out_S)

    out_E = layers.Dense(1)(chosen_hidden_states) # dot product between token representation and end vector
    out_E = layers.Flatten()(out_E)
    out_E = layers.Softmax(name="out_E")(out_E)


    model = tf.keras.models.Model(
        inputs=[input_ids, attention_mask],
        outputs = [out_S, out_E]
    )
    model.compile(tf.keras.optimizers.Adam(3e-5), 
              loss={'out_S': 'binary_crossentropy', 'out_E': 'binary_crossentropy'},
              metrics={'out_S': 'accuracy', 'out_E':'accuracy'})
    return model


### Ensemble

Train a set of DistilBert models with different parameters as base classifiers and combines their output with a sum.

In [22]:
#TODO: list of max length to be choosen
max_pairs_length_list = [480, 512]
models = [create_model(max_pairs_length_list[0]), create_model(max_pairs_length_list[1])]

TFBaseModelOutput(last_hidden_state=<KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, hidden_states=(<KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, <KerasTensor: shape=(None, 480, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>), attentions=None)
TFBaseModelOutput(last_hidden_state=<KerasTensor: shape=(None, 512, 768) dtype=float32 (created by layer 'tf_distil_bert_model')>, hidden_states=(<KerasTensor: shape=(None

In [23]:
models[0].summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 attention_mask (InputLayer)    [(None, 480)]        0           []                               
                                                                                                  
 input_ids (InputLayer)         [(None, 480)]        0           []                               
                                                                                                  
 tf_distil_bert_model (TFDistil  multiple            66362880    ['attention_mask[0][0]',         
 BertModel)                                                       'input_ids[0][0]']              
                                                                                                  
 concatenate (Concatenate)      (None, 480, 3072)    0           ['tf_distil_bert_model[0][3]'

In [24]:
models[1].summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 attention_mask (InputLayer)    [(None, 512)]        0           []                               
                                                                                                  
 input_ids (InputLayer)         [(None, 512)]        0           []                               
                                                                                                  
 tf_distil_bert_model (TFDistil  multiple            66362880    ['attention_mask[0][0]',         
 BertModel)                                                       'input_ids[0][0]']              
                                                                                                  
 concatenate_1 (Concatenate)    (None, 512, 3072)    0           ['tf_distil_bert_model[1][3

In [25]:
# Since the maximum length of the pairs is different create the corresponding dataset for the model.


# train_ds = create_data_for_dataset(small_train_dataset)
# val_ds = create_data_for_dataset(small_val_dataset)
train_ds = [create_data_for_dataset(small_train_dataset, max_pairs_length) for max_pairs_length in max_pairs_length_list]
val_ds = [create_data_for_dataset(small_val_dataset, max_pairs_length) for max_pairs_length in max_pairs_length_list]

train_ds = [train_dss.batch(BATCH_SIZE) for train_dss in train_ds]
val_ds = [val_dss.batch(BATCH_SIZE) for val_dss in val_ds]


100%|██████████| 20/20 [00:03<00:00,  6.11it/s]


Creating dataset


100%|██████████| 20/20 [00:03<00:00,  5.65it/s]


Creating dataset


100%|██████████| 5/5 [00:00<00:00,  6.92it/s]


Creating dataset


100%|██████████| 5/5 [00:00<00:00,  6.65it/s]


Creating dataset


In [26]:
def get_checkpoint_path(name):
    checkpoint_path = "training/"
    checkpoint_path = os.path.join(checkpoint_path, name, 'cp-{epoch:04d}.ckpt')
    checkpoint_dir = os.path.dirname(os.path.join('data', checkpoint_path))
    print(checkpoint_dir)
    return checkpoint_path

In [ ]:

for i, max_pairs_length in enumerate(max_pairs_length_list):
    model = models[i]
    ### COMMENT THIS CELL IF TRAINING IS ALREADY DONE ###
    checkpoint_path = get_checkpoint_path(name)
    cp_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath = checkpoint_path,
        verbose=1,
        save_weights_only = True,
        save_best_only = True
    )

    model.fit(
        train_ds[i], 
        validation_data=val_ds[i],
        epochs=1, 
        callbacks=[cp_callback]
        )

In [27]:
def start_end_token_from_probabilities(pstartv: np.array, 
                                       pendv: np.array, 
                                       dim:int=512) -> List[List[int]]:
    '''
    Returns a List of [StartToken, EndToken] elements computed from the batch outputs.
    '''
    idxs = []
    for i in range(pstartv.shape[0]):
        pstart = np.stack([pstartv[i,:]]*dim, axis=1)
        pend = np.stack([pendv[i,:]]*dim, axis=0)
        sums = pstart + pend
        sums = np.triu(sums, k=1) # Zero out lower triangular matrix + diagonal
        val = np.argmax(sums)
        row = val // dim
        col = val - dim*row
        idxs.append([row,col])
    return idxs

In [ ]:
def get_k_top_probabilities(parrays, k=10):
    sorted_arrays = []
    for parray in parrays:
        sorted_arrays.append(np.argsort(parray)[::-1][:k])
    return sorted_arrays

In [28]:
def get_index(probabilities, starting_from:int=None):
    # Get the index corresponding to the maximum probability
    if starting_from:
        # constraint to avoid that the end index is before the starting index
        probabilities = probabilities[starting_from:]
        return np.argmax(probabilities) + starting_from
    else:
        return np.argmax(probabilities)


In [34]:
batch_list = [list(train_ds[0].take(1))[0], list(train_ds[1].take(1))[0]]

random_in_batch = np.random.randint(0, BATCH_SIZE-1)

real_start = np.argmax(batch_list[0][1]["out_S"][random_in_batch])
real_end = np.argmax(batch_list[0][1]["out_E"][random_in_batch])
real_limits = [real_start, real_end]

print("Real limits: ", real_limits)

probabilities_s = [0] * (max(max_pairs_length_list))
probabilities_e = [0] * (max(max_pairs_length_list))

for i, model in enumerate(models):
    batch = batch_list[i]
    predicted_probability = model.predict(batch[0])
    predicted_limits = start_end_token_from_probabilities(*predicted_probability, max_pairs_length_list[i])[random_in_batch]
    for x in range(len(probabilities_s)):
        probabilities_s[x] = probabilities_s[x] + predicted_probability[0][random_in_batch][x] if x < max_pairs_length_list[i] else  probabilities_s[i]
        probabilities_e[x] = probabilities_e[x] +  predicted_probability[1][random_in_batch][x] if x < max_pairs_length_list[i] else  probabilities_e[i]

    print("Predicted limits: ",predicted_limits)

s = get_index(probabilities_s)
e = get_index(probabilities_e, s)
print("Predicted limits with ensemble:",[s,e])


Real limits:  [170, 171]
Predicted limits:  [39, 133]
Predicted limits:  [171, 177]
Predicted limits with ensemble: [39, 304]
